# Phishing Detection — Run Experiments on Google Colab (GPU)

This notebook runs the project's experiment scripts on a **Google Colab GPU runtime**.
It does **not** duplicate the modeling code: it clones the repository and calls the
same `scripts/*.py` CLIs documented in [README.md](../README.md), so results here
match the project docs (`docs/RESULTS.md`, `docs/EXPERIMENT_JOURNEY.md`).

**Before running:** in the Colab menu, go to `Runtime > Change runtime type` and
select **GPU** (T4 is enough for this dataset's size). CPU-only also works — every
cell below falls back to CPU automatically if no GPU is detected.

Contents:
1. Setup (clone, `uv`, install, GPU check, start MLflow UI)
2. Get the dataset
3. Run the main experiments (boosters, blend, threshold, NN embedding)
4. How to modify a model or add a new one (guide, file by file)
5. Persist trained models back to Google Drive (optional)

## 1. Setup

### 1.1 Clone the repository

Replace `<YOUR_REPO_URL>` below with the Git URL of your published repository
(GitHub/GitLab). This project is not yet published anywhere — push it first,
then paste the clone URL here. If you only have the project as a local folder
with no remote yet, run (on your machine, not in Colab):

```bash
git init
git add -A
git commit -m "Initial commit"
git remote add origin <YOUR_REPO_URL>
git push -u origin main
```

In [ ]:
REPO_URL = "<YOUR_REPO_URL>"  # e.g. https://github.com/<user>/<repo>.git
PROJECT_DIR = "/content/phishing-detection"

import os

if not os.path.isdir(PROJECT_DIR):
    if REPO_URL.startswith("<"):
        raise RuntimeError(
            "Set REPO_URL to your repository's clone URL before running this cell. "
            "See the markdown cell above for how to publish the repo first."
        )
    !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    print(f"{PROJECT_DIR} already exists, skipping clone (use 'git -C ... pull' to update).")

%cd $PROJECT_DIR

### 1.2 Install `uv` and project dependencies

The project uses **`uv`** as its only package manager (never `pip` directly — see
`CLAUDE.md`). Colab ships Python 3.12+ runtimes; `uv` will create its own managed
virtualenv so this does not fight Colab's preinstalled packages.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
# Pin Python 3.12 (optbinning/ortools have no 3.13 wheels yet — see CLAUDE.md).
!uv python install 3.12
!uv sync --python 3.12

### 1.3 GPU check

Confirms Colab handed you a GPU and that TensorFlow can see it. XGBoost and
CatBoost detect the GPU **at training time on their own** (see §4.3) — nothing to
configure here for them, this cell is only a sanity check. LightGBM is checked too,
but expect it to report no CUDA build (explained in §4.3 — this is normal).

In [ ]:
!nvidia-smi || echo "No GPU detected — Runtime > Change runtime type > GPU. Falling back to CPU is fine, just slower."

In [ ]:
%%bash
uv run python - <<'PY'
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow GPUs:", gpus)
for g in gpus:
    # Avoid TF grabbing all GPU memory upfront; plays nicer with a shared Colab GPU.
    tf.config.experimental.set_memory_growth(g, True)

from phishing.models._common import gpu_available, lightgbm_cuda_available
print("Booster GPU auto-detect (nvidia-smi):", gpu_available())
print("LightGBM CUDA build:", lightgbm_cuda_available(),
      "(False = PyPI wheel has no CUDA; lightgbm trains on CPU, which is",
      "correct behaviour, not a bug -- see SS4.3)")
PY

### 1.4 Start MLflow UI — watch training and the random-search winner live

Start the UI **now**, before launching any experiment in §3 with `--mlflow`, so you
can watch runs land in real time instead of inspecting logs after the fact.

Tracking uses the project's local file store (`./mlruns`, see
`src/phishing/observability/tracking.py`). The UI server requires
`MLFLOW_ALLOW_FILE_STORE=true` because MLflow now treats the filesystem backend as
deprecated ("maintenance mode") and refuses to start without that opt-out — this
is expected, not a sign anything is broken.

Each model's run logs its **`best_params_`** from the `RandomizedSearchCV` (the
winning hyperparameter combination for that booster) as MLflow params, plus the
CV/val/test PR-AUC and the full metric set — open a run in the UI to see them, or
sort the experiment table by `test_pr_auc` to compare models/feature modes at a
glance.

In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"  # the file:./mlruns backend is in MLflow's "maintenance mode"

!MLFLOW_ALLOW_FILE_STORE=true nohup uv run mlflow ui --backend-store-uri file:./mlruns --host 0.0.0.0 --port 5000 > mlflow_ui.log 2>&1 &
import time; time.sleep(5)
!tail -n 5 mlflow_ui.log
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(5000)"))

## 2. Get the dataset

Two options — use whichever matches where your data lives.

**Option A — already in the repo.** If `data/email_phishing_data.csv` was committed
(check `.gitignore` — by default `data/*.csv` is ignored), it is already cloned.

**Option B — upload or mount Google Drive.** Use one of the two cells below.

In [ ]:
# Option B1: direct upload (small/medium files).
# from google.colab import files
# uploaded = files.upload()  # pick email_phishing_data.csv
# !mkdir -p data && mv email_phishing_data.csv data/

In [ ]:
# Option B2: mount Google Drive and copy/symlink the CSV from there.
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p data && cp "/content/drive/MyDrive/<path-to>/email_phishing_data.csv" data/

In [ ]:
CSV = "data/email_phishing_data.csv"
import os
assert os.path.exists(CSV), f"{CSV} not found — run one of the Option B cells above."
!wc -l $CSV

## 3. Run the main experiments

These are the **same CLIs** as in the README — nothing Colab-specific about the
arguments. The only difference from running locally is speed: with a GPU runtime,
XGBoost/CatBoost train on GPU automatically (§4.3 explains how, including why
LightGBM is the exception). If you started the MLflow UI in §1.4, pass `--mlflow`
below to watch each run (and its winning hyperparameters) land live.

**Best-params cache.** The first time a (model, feature mode, search config,
training columns) combination runs, its winning hyperparameters are saved to
`best_params/<hash>.json`. Any later run with that exact combination — same
search method/budget, same features — skips `RandomizedSearchCV`/`GridSearchCV`
entirely and fits once with the cached winner, which is the main speedup for
**re-running** an experiment (a fresh combination still searches normally the
first time). Pass `--force-search` to ignore the cache and search again (e.g.
after intentionally widening a `param_distributions()` — note a wider space
still produces the *same* cache key unless `--n-iter` also changes, so
`--force-search` is the deliberate override in that case).

**Trained-embedding cache** (`scripts/best_model_report.py` only). The NN
embedding itself is the other expensive fixed cost — several minutes regardless
of which boosters run afterwards. It is cached the same way, to
`embeddings/<hash>/`, keyed by every embedding hyperparameter (`--embedding-dim`,
`--embedding-hidden1-dim`, `--embedding-hidden2-dim`, `--embedding-dropout1`,
`--embedding-dropout2`, `--embedding-patience`, `--periodic`,
`--cosine-schedule`) plus the input feature columns and training
row count. Changing the **decision threshold** (`--threshold-mode`, `--fn-cost`,
`--fp-cost`) does **not** affect either cache key — it only changes where the
already-trained models' probabilities are cut, so re-running with a different
`--fn-cost` reuses both caches and finishes in roughly the time it takes to
re-evaluate the metrics, not retrain. Pass `--force-embedding-search` to retrain
the embedding even if a matching cached one exists.

### 3.1 Quick comparison — default boosters, engineered features

Mirrors `uv run python scripts/run_experiments.py --csv data/email_phishing_data.csv`.
Add `--sample 100000` for a fast iteration pass before committing to the full dataset.
`--no-mlflow` disables logging if you skipped §1.4.

In [ ]:
!uv run python scripts/run_experiments.py \
    --csv data/email_phishing_data.csv \
    --sample 100000 \
    --search random --n-iter 40 --stacking

### 3.2 Best-model report — tuned blend + frozen NN embedding (full dataset)

Mirrors `uv run python scripts/best_model_report.py`. This is the heaviest single
run in the project (trains the NN embedding once, then 3 tuned boosters + blend on
the full 524k rows) — this is where a GPU runtime helps the most. Pass `--mlflow`
to log the run (see §1.4 to watch it live).

In [ ]:
!uv run python scripts/best_model_report.py \
    --csv data/email_phishing_data.csv \
    --search random --n-iter 40 --stacking --mlflow

### 3.3 Other experiment scripts

All scripts under `scripts/` work the same way; uncomment the one you need.
See each script's module docstring (`scripts/<name>.py`) for its full set of flags.

In [ ]:
# Per-model embedding ablation (with vs without the frozen NN embedding):
# !uv run python scripts/per_model_embedding.py --csv data/email_phishing_data.csv

# Embedding architecture sweep (dropout x embedding width):
# !uv run python scripts/embedding_arch_sweep.py --csv data/email_phishing_data.csv \
#     --sample 100000 --embedding-dim 32 --hidden2-dim 32 --dropouts 0.5 0.6 0.7 0.75

# Headless train + save a versioned model, then run inference:
# !uv run python scripts/cli.py train --csv data/email_phishing_data.csv \
#     --model xgboost --feature-mode engineered --save-best
# !uv run python scripts/cli.py infer --csv data/email_phishing_data.csv --out predictions.csv

## 4. How to modify a model or add a new one

Per `CLAUDE.md` §3.2, **each algorithm lives in its own file** under
`src/phishing/models/`, and every file exposes the same small interface so the
experiment runner, the Streamlit UI, and these notebook cells stay agnostic to which
algorithm is plugged in:

```python
NAME = "my_model"                       # registry key / CLI --model value

def build(feature_mode: str = "raw", y=None, embedding_kwargs: dict | None = None):
    """Return an UNFITTED sklearn Pipeline (feature steps + the estimator)."""
    estimator = SomeClassifier(...)
    return make_pipeline(estimator, feature_mode)   # from ._common

def param_grid() -> dict:
    """GridSearchCV grid, keys prefixed 'model__' (the Pipeline step name)."""
    return {"model__some_param": [a, b]}

def param_distributions() -> dict:      # optional, only needed for --search random
    """Wider RandomizedSearchCV space, scipy.stats distributions."""
    return {"model__some_param": uniform(0, 1)}
```

`make_pipeline()` (in `src/phishing/models/_common.py`) prepends the feature steps
implied by `feature_mode` (FeatureEngineer / encodings / NN embedding / etc.) ahead
of your estimator, so your model file only deals with the estimator itself.

Everything below edits files **inside the cloned repo** (`/content/phishing-detection`)
using a plain Python heredoc, so the change is visible to every `uv run` cell above
for the rest of the session. To persist a change, commit and push it from Colab
(§4.5) or copy it back to your machine.

### 4.1 Example — change an existing model's hyperparameter grid

Each booster file (`lightgbm_model.py`, `xgboost_model.py`, `catboost_model.py`)
has a `build()` (the unfitted pipeline) and `param_grid()` / `param_distributions()`
(the search spaces). To widen, say, XGBoost's `max_depth` range, open
`src/phishing/models/xgboost_model.py` in the Colab file browser (left sidebar ->
folder icon -> navigate to the path) and edit `param_distributions()` directly, or
do it from a cell:

In [ ]:
%%bash
python - <<'PY'
from pathlib import Path

path = Path("src/phishing/models/xgboost_model.py")
text = path.read_text()
text = text.replace(
    '"model__max_depth": randint(3, 10),',
    '"model__max_depth": randint(3, 14),  # widened from 10 on Colab GPU',
)
path.write_text(text)
print("Updated max_depth range in xgboost_model.py")
PY

Re-run §3.1/3.2 and the new grid is picked up automatically — no registry change
needed when you're only editing an existing file's `build()`/`param_*()` bodies.

**Note on the best-params cache (§3):** the cache key is based on the model name,
feature mode, search method/budget and training columns — it does **not** hash
the contents of `param_distributions()`. So after widening a range like above,
also pass `--force-search` once to make sure the new range is actually explored,
otherwise a previously cached winner (found under the old, narrower range) is
reused silently.

### 4.2 Example — add a brand-new model file

To add a new algorithm (say, a `BalancedRandomForest`-style variant), create a new
file following the same shape, then register it. Two steps:

1. Create `src/phishing/models/my_new_model.py` implementing `NAME`, `build()`,
   `param_grid()` (and optionally `param_distributions()`).
2. Register it in `src/phishing/models/__init__.py`: import the module and add
   `my_new_model.NAME: my_new_model` to `ALL_MODELS` (add to `DEFAULT_MODELS` too
   only if it should run by default alongside lightgbm/xgboost/catboost).

In [ ]:
%%bash
cat > src/phishing/models/extratrees_model.py <<'PY'
"""ExtraTrees — example new model added from the Colab notebook."""

# %%
from __future__ import annotations

from scipy.stats import randint
from sklearn.ensemble import ExtraTreesClassifier

from ._common import make_pipeline

NAME = "extratrees"


# %%
def build(feature_mode: str = "raw", y=None, embedding_kwargs: dict | None = None):
    estimator = ExtraTreesClassifier(
        class_weight="balanced", n_jobs=-1, random_state=42,
    )
    return make_pipeline(estimator, feature_mode)


# %%
def param_grid() -> dict:
    return {"model__n_estimators": [200, 400], "model__max_depth": [None, 10]}


# %%
def param_distributions() -> dict:
    return {
        "model__n_estimators": randint(200, 800),
        "model__max_depth": randint(4, 30),
    }
PY
echo "Created src/phishing/models/extratrees_model.py"

In [ ]:
%%bash
python - <<'PY'
from pathlib import Path

path = Path("src/phishing/models/__init__.py")
text = path.read_text()
text = text.replace(
    "from . import (\n    adaboost_model,",
    "from . import (\n    adaboost_model,\n    extratrees_model,",
)
text = text.replace(
    "ALL_MODELS = {\n    lightgbm_model.NAME: lightgbm_model,",
    "ALL_MODELS = {\n    lightgbm_model.NAME: lightgbm_model,\n    extratrees_model.NAME: extratrees_model,",
)
path.write_text(text)
print("Registered extratrees in ALL_MODELS")
PY

In [ ]:
# Sanity check + a quick run on the new model.
!uv run python -c "from phishing.models import ALL_MODELS; print('extratrees' in ALL_MODELS)"
!uv run python scripts/run_experiments.py --csv data/email_phishing_data.csv \
    --sample 50000 --models extratrees

### 4.3 Where GPU support lives (so you can extend it)

`src/phishing/models/_common.py` exposes `gpu_available()` (cheap `nvidia-smi`
probe) and `lightgbm_cuda_available()`. `lightgbm_model.py`, `xgboost_model.py` and
`catboost_model.py` call the relevant check inside `build()` and set the GPU param
only when it is actually usable:

| model | GPU param set when... |
|---|---|
| LightGBM | `device="cuda"`, only if `lightgbm_cuda_available()` |
| XGBoost | `device="cuda"` (with `tree_method="hist"`), if `gpu_available()` |
| CatBoost | `task_type="GPU"`, if `gpu_available()` |
| TensorFlow (NN embedding / `tensorflow_dnn`) | automatic — TF uses any visible GPU with no code change; see the memory-growth cell in §1.3 |

**Why LightGBM needs its own check.** The default PyPI/`uv` wheel of LightGBM ships
**without CUDA support** — only the generic OpenCL backend (`device="gpu"`). On a
machine with *both* a discrete NVIDIA GPU and an integrated GPU (common on laptops,
less likely on a Colab VM but possible on your own machine), OpenCL can silently
pick the integrated GPU — slower than CPU, not faster, and not the device you
asked for. `lightgbm_cuda_available()` does a 1-row trial fit with
`device="cuda"` and only enables GPU training if that real CUDA path works;
otherwise LightGBM trains on CPU, which is the **correct, faster choice** on an
unpatched wheel, not a fallback to apologise for. (Building LightGBM from source
with `-DUSE_CUDA=1` would unlock the real GPU path, but that is out of scope for a
plain `uv sync` install.) The probe's result is process-cached, and the one-time
`[LightGBM] [Fatal] CUDA Tree Learner was not enabled...` line it prints during
that trial fit is expected — it is caught and turned into `False`, not a crash.

**Why the hyperparameter search runs sequentially on a GPU-capable booster.** A GPU
is a single shared device. `RandomizedSearchCV`/`GridSearchCV`'s default
`n_jobs=-1` spawns one CPU process per CV candidate — fine on CPU, but on GPU it
means several processes fighting over one device's context instead of actually
parallelising, which is *slower* than plain CPU, not faster.
`src/phishing/experiments/runner.py` (`run_model`) handles this per model: it picks
`n_jobs=1` for the search **only** when that specific booster's GPU check
(`lightgbm_cuda_available()` / `gpu_available()`) returns true, so a GPU-capable
booster gets the device exclusively, one candidate at a time, while CPU-only models
(or LightGBM without a CUDA build) keep full CPU parallelism.

There is **no separate `_gpu` model file** — the same `lightgbm`/`xgboost`/`catboost`
names work on CPU and GPU transparently, picking the fastest real option available.
To force CPU even on a GPU runtime (e.g. to reproduce the CPU numbers in the docs),
set the environment variable before running any script:

```python
import os; os.environ["PHISHING_FORCE_CPU"] = "1"
```

To add GPU support to a model that doesn't have it yet, follow the same pattern:
import the relevant check from `._common`, register it in `runner.py`'s
`_GPU_CHECK` mapping (so the search picks the right `n_jobs`), and conditionally
add that library's GPU flag to the estimator's constructor kwargs inside
`build()` — and verify with a real timing comparison, not just "no error was
raised", before trusting it.

In [ ]:
# Force CPU even on a GPU runtime, e.g. to compare against the CPU numbers in docs/RESULTS.md.
# import os
# os.environ["PHISHING_FORCE_CPU"] = "1"

### 4.4 Editing the NN embedding / TensorFlow architecture

The dense net used by both `tensorflow_dnn` and the reusable NN embedding lives in
`src/phishing/models/_tf_net.py` (`build_keras_model`), wired up in
`src/phishing/features/nn_embedding.py`. To experiment with the architecture (layer
widths, dropout, embedding dimension) without editing files by hand, pass the
existing parameters through the scripts — e.g. `scripts/embedding_arch_sweep.py`
already exposes `--embedding-dim`, `--hidden2-dim`, `--dropouts`
(see §3.3). For structural changes (new layers, a different skip-connection), edit
`_tf_net.py`'s `build_keras_model()` directly — it runs on GPU automatically once
TensorFlow detects one (§1.3), no Colab-specific code needed there.

**Early-stopping patience.** `NNEmbedding(patience=...)` controls how many epochs
without a `val_pr_auc` improvement are tolerated before training stops (default
**50**, monitored in `nn_embedding.py`). A higher patience lets the embedding run
longer in search of a better validation score before giving up — useful on the full
dataset where the PR-AUC curve can still improve slowly past epoch 150–200; it does
not change the architecture, only how long `fit()` is willing to keep training.

### 4.5 Run the test suite after a change

Before trusting a modified or new model, run the project's smoke tests — they
exercise the full fit/predict/save/load path on synthetic data and stay fast.

In [ ]:
!uv run pytest -q

## 5. Persist results back to Google Drive (optional)

Colab runtimes are ephemeral — anything in `/content` is lost when the session
recycles. To keep trained model versions (`models/`) or MLflow runs (`mlruns/`),
copy them to a mounted Drive, or commit code changes and push to your repo.

Training also leaves working-directory clutter that `.gitignore` already excludes
from any `git add -A`: CatBoost's `catboost_info/` directory (created next to
wherever the process runs, even with `allow_writing_files=False` set on the
estimator itself — that flag only stops CatBoost's *own* file outputs, not this
directory), the local `mlflow_ui.log` / training `*.log` files from §1.4/§3, and a
stray `mlflow.db` if `mlflow ui` is ever started without `--backend-store-uri
file:./mlruns`. Nothing to do here — just don't be surprised to see them locally.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/phishing-detection-runs"
# !cp -r models mlruns "/content/drive/MyDrive/phishing-detection-runs/" 2>/dev/null || true

In [ ]:
# Push a code change made in this notebook back to your repository.
# !git config user.email "you@example.com"
# !git config user.name "Your Name"
# !git add -A
# !git commit -m "Experiment from Colab"
# !git push origin main